
# AI in Fluids - Week 2 Colab
## Neural networks and rarefied-flow surrogates

This notebook supports the Week-2 lecture.  It is designed to make a neural network less mysterious.

You will:

1. compute a one-neuron prediction by hand in Python,
2. perform one gradient-descent update,
3. plot activation functions,
4. generate a synthetic rarefied-flow dataset,
5. train simple baselines and a small DNN,
6. evaluate numerical errors,
7. perform physical validation checks.

The rarefied-flow dataset in this notebook is synthetic.  It is a teaching model, not a substitute for DSMC, Boltzmann, or experimental data.



## 0. Imports

Run this cell first.  Google Colab normally includes TensorFlow, NumPy, scikit-learn, pandas, and Matplotlib.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(7)
tf.random.set_seed(7)

plt.rcParams.update({"figure.figsize": (6,4), "font.size": 11})
print("TensorFlow version:", tf.__version__)



## 1. One neuron: forward pass

A single linear-output neuron is

\begin{equation}
o = w_1 x_1 + w_2 x_2 + b,
\end{equation}

where:

- \(o\) is the model output,
- \(x_1 = \log_{10}(Kn)\) is a logarithmic Knudsen-number feature,
- \(x_2 = \alpha\) is the accommodation coefficient,
- \(w_1,w_2\) are weights,
- \(b\) is the bias.

Use the numerical values from the lecture.


In [ ]:

# Features and target for one training sample
x1 = -2.0       # log10(Kn), so Kn = 1e-2
x2 = 0.8        # accommodation coefficient alpha
t = 0.25        # target normalized mass-flow correction

# Initial parameters
w1 = 0.20
w2 = -0.10
b = 0.05

# Forward pass
o = w1*x1 + w2*x2 + b
e = t - o
C = 0.5*(t - o)**2

print(f"prediction o = {o:.5f}")
print(f"error e = t - o = {e:.5f}")
print(f"loss C = 0.5*(t-o)^2 = {C:.5f}")



## 2. One gradient-descent update

For this linear-output neuron with squared-error loss,

\begin{equation}
w_j \leftarrow w_j + \eta (t-o)x_j, \qquad b \leftarrow b + \eta(t-o),
\end{equation}

where \(\eta\) is the learning rate and \(j\) is the feature index.


In [ ]:

eta = 0.05  # learning rate

w1_new = w1 + eta*(t-o)*x1
w2_new = w2 + eta*(t-o)*x2
b_new  = b  + eta*(t-o)

o_new = w1_new*x1 + w2_new*x2 + b_new
C_new = 0.5*(t - o_new)**2

print("Old parameters:", w1, w2, b)
print("New parameters:", w1_new, w2_new, b_new)
print(f"old prediction = {o:.5f}, old loss = {C:.5f}")
print(f"new prediction = {o_new:.5f}, new loss = {C_new:.5f}")



### Question
Did the prediction move closer to the target?  Explain in one sentence in your report.



## 3. Activation functions

Activation functions introduce nonlinearity.  Without nonlinear activations, stacked layers reduce to one linear map.


In [ ]:

z = np.linspace(-5, 5, 500)
sigmoid = 1/(1 + np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0, z)
linear = z

plt.figure()
plt.plot(z, sigmoid, label="sigmoid")
plt.plot(z, tanh, label="tanh")
plt.plot(z, relu, label="ReLU")
plt.plot(z, linear, label="linear")
plt.ylim(-2, 5)
plt.xlabel("pre-activation z")
plt.ylabel("activation output")
plt.title("Common activation functions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 4. Synthetic rarefied-flow dataset

We use the synthetic relation

\begin{equation}
Q^* = \left(1-\frac{1}{\Pi}\right)
\left[1 + C_s\frac{2-\alpha}{\alpha}\frac{Kn}{1+Kn}\right]
\Theta^{-1/2}
\left[1+C_r\tanh\left(\log_{10}Kn+1\right)\right] + \epsilon.
\end{equation}

Definitions:

- \(Q^*\): normalized mass-flow proxy,
- \(Kn\): Knudsen number,
- \(\alpha\): accommodation coefficient,
- \(\Pi = p_{in}/p_{out}\): pressure ratio,
- \(\Theta = T_w/T_0\): wall-to-reference temperature ratio,
- \(C_s\): slip-strength coefficient,
- \(C_r\): rarefaction-transition coefficient,
- \(\epsilon\): small noise term.


In [ ]:

def synthetic_Q(log10_Kn, alpha, Pi, Theta, noise_std=0.01, seed=None):
    rng = np.random.default_rng(seed)
    Kn = 10**log10_Kn
    Cs = 1.25
    Cr = 0.10
    clean = (1 - 1/Pi) * (1 + Cs*((2-alpha)/alpha)*(Kn/(1+Kn))) * (Theta**(-0.5)) * (1 + Cr*np.tanh(log10_Kn + 1))
    noise = rng.normal(0.0, noise_std, size=np.shape(clean))
    return clean + noise

N = 5000
rng = np.random.default_rng(42)

log10_Kn = rng.uniform(-4.0, 1.5, N)       # Kn from 1e-4 to about 31.6
alpha = rng.uniform(0.6, 1.0, N)           # accommodation coefficient
Pi = rng.uniform(1.1, 5.0, N)              # pressure ratio p_in / p_out
Theta = rng.uniform(0.7, 1.5, N)           # wall/reference temperature ratio

Qstar = synthetic_Q(log10_Kn, alpha, Pi, Theta, noise_std=0.01, seed=5)

df = pd.DataFrame({
    "log10_Kn": log10_Kn,
    "Kn": 10**log10_Kn,
    "alpha": alpha,
    "Pi": Pi,
    "Theta": Theta,
    "Qstar": Qstar
})

df.head()



## 5. Classify rarefaction regimes

Use the approximate ranges:

- continuum: \(Kn < 10^{-3}\),
- slip: \(10^{-3} \le Kn < 10^{-1}\),
- transition: \(10^{-1} \le Kn < 10\),
- free molecular: \(Kn \ge 10\).


In [ ]:

def classify_regime(Kn):
    Kn = np.asarray(Kn)
    labels = np.empty(Kn.shape, dtype=object)
    labels[Kn < 1e-3] = "continuum"
    labels[(Kn >= 1e-3) & (Kn < 1e-1)] = "slip"
    labels[(Kn >= 1e-1) & (Kn < 10)] = "transition"
    labels[Kn >= 10] = "free molecular"
    return labels

df["regime"] = classify_regime(df["Kn"].values)
df["regime"].value_counts()


In [ ]:

plt.figure()
for reg, g in df.groupby("regime"):
    plt.scatter(g["log10_Kn"], g["Qstar"], s=8, alpha=0.35, label=reg)
plt.xlabel("log10(Kn)")
plt.ylabel("Q*")
plt.title("Synthetic rarefied-flow data by regime")
plt.legend(markerscale=2)
plt.grid(True, alpha=0.3)
plt.show()



## 6. Feature matrix and target vector

The feature matrix is

\begin{equation}
X = [\log_{10}Kn,\alpha,\Pi,\Theta].
\end{equation}

The target vector is \(y=Q^*\).


In [ ]:

feature_cols = ["log10_Kn", "alpha", "Pi", "Theta"]
X = df[feature_cols].values.astype("float32")
y = df["Qstar"].values.astype("float32").reshape(-1, 1)

X_train, X_temp, y_train, y_temp, reg_train, reg_temp = train_test_split(
    X, y, df["regime"].values, test_size=0.30, random_state=10
)
X_val, X_test, y_val, y_test, reg_val, reg_test = train_test_split(
    X_temp, y_temp, reg_temp, test_size=0.50, random_state=10
)

scaler_X = StandardScaler().fit(X_train)
scaler_y = StandardScaler().fit(y_train)

X_train_s = scaler_X.transform(X_train)
X_val_s = scaler_X.transform(X_val)
X_test_s = scaler_X.transform(X_test)

y_train_s = scaler_y.transform(y_train)
y_val_s = scaler_y.transform(y_val)
y_test_s = scaler_y.transform(y_test)

print("train/val/test shapes:", X_train.shape, X_val.shape, X_test.shape)



## 7. Error metrics


In [ ]:

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    rel_l2 = np.linalg.norm(y_true - y_pred) / np.linalg.norm(y_true)
    return rmse, mae, rel_l2



## 8. Baseline 1: polynomial regression


In [ ]:

poly_model = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=3, include_bias=False),
    Ridge(alpha=1e-6)
)
poly_model.fit(X_train, y_train.ravel())
poly_pred = poly_model.predict(X_test).reshape(-1, 1)

poly_metrics = metrics(y_test, poly_pred)
print("Polynomial regression metrics (RMSE, MAE, RelL2):", poly_metrics)



## 9. Baseline 2: nearest-neighbor/local interpolation


In [ ]:

knn_model = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=8, weights="distance"))
knn_model.fit(X_train, y_train.ravel())
knn_pred = knn_model.predict(X_test).reshape(-1, 1)

knn_metrics = metrics(y_test, knn_pred)
print("KNN/local interpolation metrics (RMSE, MAE, RelL2):", knn_metrics)



## 10. DNN surrogate

The DNN receives four inputs and predicts one scalar output.


In [ ]:

model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(32, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1, activation="linear")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=30, restore_best_weights=True
)

history = model.fit(
    X_train_s, y_train_s,
    validation_data=(X_val_s, y_val_s),
    epochs=500,
    batch_size=64,
    callbacks=[callback],
    verbose=0
)

print("Training finished after", len(history.history["loss"]), "epochs")


In [ ]:

plt.figure()
plt.semilogy(history.history["loss"], label="training loss")
plt.semilogy(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.ylabel("MSE loss (scaled target)")
plt.title("DNN training history")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:

y_pred_s = model.predict(X_test_s, verbose=0)
y_pred = scaler_y.inverse_transform(y_pred_s)

dnn_metrics = metrics(y_test, y_pred)
print("DNN metrics (RMSE, MAE, RelL2):", dnn_metrics)



## 11. Numerical comparison


In [ ]:

summary = pd.DataFrame(
    [poly_metrics, knn_metrics, dnn_metrics],
    columns=["RMSE", "MAE", "Relative L2"],
    index=["Polynomial", "KNN/local", "DNN"]
)
summary


In [ ]:

plt.figure()
plt.scatter(y_test, poly_pred, s=12, alpha=0.5, label="Polynomial")
plt.scatter(y_test, knn_pred, s=12, alpha=0.5, label="KNN/local")
plt.scatter(y_test, y_pred, s=12, alpha=0.5, label="DNN")
lo = min(y_test.min(), y_pred.min(), poly_pred.min(), knn_pred.min())
hi = max(y_test.max(), y_pred.max(), poly_pred.max(), knn_pred.max())
plt.plot([lo, hi], [lo, hi], "k--", label="perfect")
plt.xlabel("true Q*")
plt.ylabel("predicted Q*")
plt.title("Predicted versus true values")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 12. Physical validation checks

### Check 1: positivity


In [ ]:

negative_count = np.sum(y_pred.reshape(-1) < 0)
print("Number of negative DNN predictions:", negative_count, "out of", len(y_pred))



### Check 2: pressure-ratio monotonicity

For fixed \(Kn\), \(\alpha\), and \(\Theta\), the synthetic function should increase as \(\Pi\) increases.


In [ ]:

fixed_logKn = -2.0
fixed_alpha = 0.85
fixed_Theta = 1.0
Pi_grid = np.linspace(1.1, 5.0, 100)
X_mono = np.column_stack([
    np.full_like(Pi_grid, fixed_logKn),
    np.full_like(Pi_grid, fixed_alpha),
    Pi_grid,
    np.full_like(Pi_grid, fixed_Theta)
]).astype("float32")

X_mono_s = scaler_X.transform(X_mono)
Q_mono_pred = scaler_y.inverse_transform(model.predict(X_mono_s, verbose=0)).reshape(-1)
violations = np.sum(np.diff(Q_mono_pred) < -1e-5)

plt.figure()
plt.plot(Pi_grid, Q_mono_pred, label="DNN prediction")
plt.xlabel("pressure ratio Pi")
plt.ylabel("predicted Q*")
plt.title("Monotonicity check at fixed Kn, alpha, Theta")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Number of monotonicity violations:", violations)



### Check 3: regime-wise RMSE


In [ ]:

regime_rows = []
for reg in ["continuum", "slip", "transition", "free molecular"]:
    mask = (reg_test == reg)
    if mask.sum() > 0:
        regime_rows.append((reg, mask.sum(), *metrics(y_test[mask], y_pred[mask])))

regime_table = pd.DataFrame(regime_rows, columns=["regime", "N", "RMSE", "MAE", "Relative L2"])
regime_table



## 13. Extrapolation experiment

Train only on continuum + slip data and test in transition + free-molecular regimes.  This intentionally creates a difficult extrapolation problem.


In [ ]:

mask_train_low = X[:,0] <= -1.0      # log10(Kn) <= -1, mostly continuum/slip
mask_test_high = X[:,0] > -1.0       # transition/free molecular

X_low = X[mask_train_low]
y_low = y[mask_train_low]
X_high = X[mask_test_high]
y_high = y[mask_test_high]

X_low_train, X_low_val, y_low_train, y_low_val = train_test_split(
    X_low, y_low, test_size=0.25, random_state=123
)

scaler_X_low = StandardScaler().fit(X_low_train)
scaler_y_low = StandardScaler().fit(y_low_train)

X_low_train_s = scaler_X_low.transform(X_low_train)
X_low_val_s = scaler_X_low.transform(X_low_val)
X_high_s = scaler_X_low.transform(X_high)
y_low_train_s = scaler_y_low.transform(y_low_train)
y_low_val_s = scaler_y_low.transform(y_low_val)

extrap_model = keras.Sequential([
    layers.Input(shape=(4,)),
    layers.Dense(32, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1, activation="linear")
])
extrap_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
extrap_hist = extrap_model.fit(
    X_low_train_s, y_low_train_s,
    validation_data=(X_low_val_s, y_low_val_s),
    epochs=400,
    batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)],
    verbose=0
)

y_high_pred = scaler_y_low.inverse_transform(extrap_model.predict(X_high_s, verbose=0))
print("Extrapolation metrics on log10(Kn)>-1:", metrics(y_high, y_high_pred))


In [ ]:

plt.figure()
plt.scatter(X_high[:,0], y_high, s=8, alpha=0.3, label="true high-Kn data")
plt.scatter(X_high[:,0], y_high_pred, s=8, alpha=0.3, label="DNN trained only on low-Kn")
plt.xlabel("log10(Kn)")
plt.ylabel("Q*")
plt.title("Extrapolation from continuum/slip into transition/free molecular")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### Final reflection for your report

Answer these questions:

1. Which model had the lowest test error in the random split?
2. Did the DNN behave physically in the monotonicity and positivity checks?
3. What happened in the extrapolation experiment?
4. Why is extrapolation across Knudsen regimes risky?
5. How would DSMC noise make this task harder?


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../../ARTICLE_FIGURE_MAP.md).
